
# Phase 4 — Reliable Kafka Streaming Pipeline

## Objectives
- Read taxi events from Confluent Cloud Kafka.
- Preserve raw Kafka messages in Delta Lake.
- Parse and validate incoming events.
- Route valid records and invalid records separately.
- Prevent duplicate processing.
- Test checkpointing and recovery.
- Monitor streaming progress and data quality.

## Architecture
Python Producer → Confluent Cloud Kafka (`taxi-trips`)
→ Spark Structured Streaming → Raw Delta
→ Validation → Valid Records / Quarantine

## Safety Rules
- Preserve the existing batch pipeline.
- Do not overwrite existing streaming tables.
- Do not reset or delete existing checkpoints.
- Use separate tables and checkpoint paths for Phase 4.
- Never hardcode Kafka credentials in notebook code.



In [0]:

value = dbutils.secrets.get(scope="confluent_kafka", key="api_key")
print("Secret retrieved successfully. Length:", len(value))

In [0]:
# ============================================
# 01 — Configuration
# ============================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    StringType,
    DoubleType,
)

# Kafka topic
KAFKA_TOPIC = "taxi-trips"

# Existing Kafka bootstrap server
KAFKA_BOOTSTRAP_SERVERS = "pkc-619z3.us-east1.gcp.confluent.cloud:9092"

# Confluent Cloud credentials
# Retrieve these securely from your Databricks secret scope.
KAFKA_API_KEY = dbutils.secrets.get(
    scope="confluent_kafka",
    key="api_key",
)

KAFKA_API_SECRET = dbutils.secrets.get(
    scope="confluent_kafka",
    key="api_secret",
)

# Kafka connection options
kafka_options = {
    "kafka.bootstrap.servers": KAFKA_BOOTSTRAP_SERVERS,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule "
        "required "
        f'username="{KAFKA_API_KEY}" '
        f'password="{KAFKA_API_SECRET}";'
    ),
}

print("Kafka authentication configuration updated.")

print("Configuration loaded.")
print(f"Topic: {KAFKA_TOPIC}")

In [0]:
# ============================================
# 02 — Test Kafka Connection
# ============================================

df_kafka_test = (
    spark.read
    .format("kafka")
    .options(**kafka_options)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

print("Kafka read initialized successfully.")
df_kafka_test.printSchema()

In [0]:
# ============================================
# 03 — Test Kafka Message Retrieval
# ============================================

sample_messages = (
    df_kafka_test
    .select(
        "topic",
        "partition",
        "offset",
        F.col("value").cast("string").alias("message")
    )
    .limit(1)
    .collect()
)

if sample_messages:
    print("Kafka message retrieval successful.")
    print(sample_messages[0])
else:
    print("No messages found in the selected offset range.")

In [0]:
df_kafka_test = (
    spark.read
    .format("kafka")
    .options(**kafka_options)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

print("Kafka read initialized successfully.")
df_kafka_test.printSchema()

In [0]:
from pyspark.sql import functions as F

df_kafka_sample = (
    df_kafka_test
    .select(
        "topic",
        "partition",
        "offset",
        "timestamp",
        F.col("value").cast("string").alias("raw_value")
    )
    .limit(5)
)

display(df_kafka_sample)

In [0]:
raw_table_name = "main.`nyc-taxi`.phase4_raw_kafka_messages"

if spark.catalog.tableExists(raw_table_name):
    print(f"Table already exists: {raw_table_name}")
    print("We will inspect it before making any changes.")
else:
    print(f"Table does not exist: {raw_table_name}")
    print("Safe to plan its initial creation.")

In [0]:
phase4_raw_checkpoint = (
    "/Volumes/main/nyc-taxi/nyc_taxi_data/"
    "checkpoints/phase4_raw_kafka_messages"
)

print("Target table:", raw_table_name)
print("Target checkpoint:", phase4_raw_checkpoint)
print("Existing pipeline checkpoints will not be changed.")

In [0]:
kafka_message_count = df_kafka_test.count()

print(f"Messages available from the selected offset range: {kafka_message_count:,}")
print("No Delta tables or streaming checkpoints were modified.")

In [0]:
from pyspark.sql import functions as F

df_phase4_raw = (
    spark.read
    .format("kafka")
    .options(**kafka_options)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load()
    .select(
        F.col("topic"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
        F.col("timestamp").alias("kafka_timestamp"),
        F.col("key").cast("string").alias("message_key"),
        F.col("value").cast("string").alias("raw_value"),
        F.current_timestamp().alias("captured_at")
    )
)

print("Raw-capture DataFrame prepared.")
df_phase4_raw.printSchema()

In [0]:
from pyspark.sql import functions as F

df_phase4_raw.agg(
    F.count("*").alias("total_messages"),
    F.sum(
        F.when(F.col("raw_value").isNull(), 1).otherwise(0)
    ).alias("null_payloads"),
    F.countDistinct(
        "topic", "kafka_partition", "kafka_offset"
    ).alias("unique_kafka_ids")
).show(truncate=False)

In [0]:
if spark.catalog.tableExists(raw_table_name):
    print("STOP: The target table already exists.")
    print("No data was written. Inspect the table before proceeding.")
else:
    (
        df_phase4_raw.write
        .format("delta")
        .mode("append")
        .saveAsTable(raw_table_name)
    )
    print(f"Raw capture saved successfully to {raw_table_name}")

In [0]:
df_phase4_saved = spark.table(raw_table_name)

df_phase4_saved.agg(
    F.count("*").alias("total_messages"),
    F.countDistinct(
        "topic", "kafka_partition", "kafka_offset"
    ).alias("unique_kafka_ids"),
    F.sum(
        F.when(F.col("raw_value").isNull(), 1).otherwise(0)
    ).alias("null_payloads")
).show(truncate=False)

In [0]:
spark.table("main.`nyc-taxi`.bronze_taxi_trips").printSchema()

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    LongType, DoubleType, StringType,
    IntegerType, TimestampNTZType
)

taxi_schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampNTZType(), True),
    StructField("tpep_dropoff_datetime", TimestampNTZType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", IntegerType(), True),
    StructField("airport_fee", IntegerType(), True)
])

print("Taxi JSON schema defined.")
taxi_schema.simpleString()

In [0]:
df_phase4_parsed = (
    df_phase4_saved
    .withColumn(
        "taxi",
        F.from_json(F.col("raw_value"), taxi_schema)
    )
)

df_phase4_parsed.select(
    "kafka_partition",
    "kafka_offset",
    "taxi.*"
).show(5, truncate=False)

In [0]:
df_phase4_parsed.agg(
    F.count("*").alias("total_messages"),
    F.sum(
        F.when(F.col("taxi").isNull(), 1).otherwise(0)
    ).alias("json_parse_failures"),
    F.sum(
        F.when(
            F.col("taxi.VendorID").isNull(), 1
        ).otherwise(0)
    ).alias("null_vendor_id")
).show(truncate=False)

In [0]:
df_phase4_parsed.filter(
    F.col("taxi.VendorID").isNull()
).select(
    "topic",
    "kafka_partition",
    "kafka_offset",
    "raw_value",
    "taxi.*"
).show(truncate=False)

In [0]:
df_phase4_saved.filter(
    (F.col("kafka_partition") == 1) &
    (F.col("kafka_offset").isin(0, 38))
).select(
    "topic",
    "kafka_partition",
    "kafka_offset",
    "raw_value"
).show(truncate=False)

In [0]:
df_phase4_saved.filter(
    (F.col("kafka_partition") == 1) &
    F.col("kafka_offset").between(0, 42)
).select(
    "kafka_partition",
    "kafka_offset",
    "raw_value"
).orderBy("kafka_offset").show(50, truncate=80)

In [0]:
df_phase4_parsed.select(
    "kafka_partition",
    "kafka_offset",
    "raw_value",
    F.from_json(
        F.col("raw_value"),
        taxi_schema
    ).isNull().alias("json_parse_failed")
).filter(
    F.col("json_parse_failed")
).show(truncate=False)

In [0]:
df_phase4_parsed.filter(
    F.col("taxi.VendorID").isNull()
).select(
    "kafka_partition",
    "kafka_offset",
    "raw_value",
    F.col("taxi").isNull().alias("parsed_struct_is_null"),
    F.col("taxi.VendorID").isNull().alias("vendor_id_is_null")
).show(truncate=False)

In [0]:
df_phase4_classified = (
    df_phase4_parsed
    .withColumn(
        "validation_status",
        F.when(
            F.col("taxi.VendorID").isNull(),
            F.lit("QUARANTINE_MISSING_REQUIRED_FIELDS")
        ).otherwise(F.lit("VALID"))
    )
)

df_phase4_classified.groupBy(
    "validation_status"
).count().show(truncate=False)

In [0]:
phase4_valid_table = "main.`nyc-taxi`.phase4_valid_taxi_events"
phase4_quarantine_table = "main.`nyc-taxi`.phase4_quarantine_taxi_events"

for table_name in [phase4_valid_table, phase4_quarantine_table]:
    exists = spark.catalog.tableExists(table_name)
    print(f"{table_name}: {'EXISTS' if exists else 'DOES NOT EXIST'}")

In [0]:
df_phase4_valid = (
    df_phase4_classified
    .filter(F.col("validation_status") == "VALID")
    .select(
        "topic",
        "kafka_partition",
        "kafka_offset",
        "kafka_timestamp",
        "message_key",
        "raw_value",
        "captured_at",
        "taxi.*"
    )
)

df_phase4_quarantine = (
    df_phase4_classified
    .filter(
        F.col("validation_status") != "VALID"
    )
    .select(
        "topic",
        "kafka_partition",
        "kafka_offset",
        "kafka_timestamp",
        "message_key",
        "raw_value",
        "captured_at",
        F.col("validation_status").alias("quarantine_reason")
    )
)

print("Valid records:", df_phase4_valid.count())
print("Quarantined records:", df_phase4_quarantine.count())

In [0]:
if spark.catalog.tableExists(phase4_valid_table):
    print("STOP: Valid-events table already exists. No data written.")
else:
    (
        df_phase4_valid.write
        .format("delta")
        .mode("append")
        .saveAsTable(phase4_valid_table)
    )
    print(f"Saved valid records to {phase4_valid_table}")

In [0]:
if spark.catalog.tableExists(phase4_quarantine_table):
    print("STOP: Quarantine table already exists. No data written.")
else:
    (
        df_phase4_quarantine.write
        .format("delta")
        .mode("append")
        .saveAsTable(phase4_quarantine_table)
    )
    print(f"Saved quarantined records to {phase4_quarantine_table}")

In [0]:
table_counts = [
    (raw_table_name, "RAW"),
    (phase4_valid_table, "VALID"),
    (phase4_quarantine_table, "QUARANTINE")
]

for table_name, label in table_counts:
    count = spark.table(table_name).count()
    print(f"{label}: {count:,} records")

In [0]:
raw_df = spark.table(raw_table_name)
valid_df = spark.table(phase4_valid_table)
quarantine_df = spark.table(phase4_quarantine_table)

key_cols = ["topic", "kafka_partition", "kafka_offset"]

raw_keys = raw_df.select(*key_cols).distinct()
valid_keys = valid_df.select(*key_cols).distinct()
quarantine_keys = quarantine_df.select(*key_cols).distinct()

output_keys = valid_keys.unionByName(quarantine_keys)

missing = raw_keys.join(output_keys, key_cols, "left_anti").count()
unexpected = output_keys.join(raw_keys, key_cols, "left_anti").count()
overlap = valid_keys.join(quarantine_keys, key_cols, "inner").count()

print("Raw unique messages:", raw_keys.count())
print("Output unique messages:", output_keys.distinct().count())
print("Missing messages:", missing)
print("Unexpected messages:", unexpected)
print("Messages in both output tables:", overlap)

In [0]:
from pyspark.sql import functions as F

tables_to_check = {
    "RAW": raw_table_name,
    "VALID": phase4_valid_table,
    "QUARANTINE": phase4_quarantine_table
}

key_cols = ["topic", "kafka_partition", "kafka_offset"]

for label, table_name in tables_to_check.items():
    df = spark.table(table_name)

    duplicate_count = (
        df.groupBy(*key_cols)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print(f"{label} duplicate Kafka identities: {duplicate_count}")
    

In [0]:
business_key = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "total_amount"
]

valid_df = spark.table(phase4_valid_table)

duplicate_trip_groups = (
    valid_df
    .groupBy(*business_key)
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Repeated business-key groups:",
    duplicate_trip_groups.count()
)

duplicate_trip_groups.orderBy(
    F.desc("count")
).show(10, truncate=False)

In [0]:
duplicate_trip_groups.groupBy("count").count().orderBy("count").show()

In [0]:
(
    valid_df
    .filter(
        (F.col("VendorID") == 1)
        & (F.col("tpep_pickup_datetime") == F.to_timestamp_ntz(
            F.lit("2016-01-01 00:41:58")
        ))
        & (F.col("tpep_dropoff_datetime") == F.to_timestamp_ntz(
            F.lit("2016-01-01 01:22:06")
        ))
        & (F.col("trip_distance") == 13.8)
        & (F.col("total_amount") == 44.3)
    )
    .select(
        "topic",
        "kafka_partition",
        "kafka_offset",
        "message_key",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_distance",
        "total_amount"
    )
    .orderBy("kafka_partition", "kafka_offset")
    .show(truncate=False)
)

In [0]:
print(
    "Total valid records:",
    valid_df.count()
)

print(
    "Distinct business-key groups:",
    valid_df.select(*business_key).distinct().count()
)

print(
    "Distinct Kafka message keys:",
    valid_df.select("message_key").distinct().count()
)

In [0]:
(
    valid_df
    .groupBy("message_key")
    .agg(
        F.count("*").alias("record_count"),
        F.countDistinct(
            F.struct(*[F.col(c) for c in business_key])
        ).alias("distinct_trips")
    )
    .orderBy("message_key")
    .show(20, truncate=False)
)

In [0]:
key_mapping = (
    valid_df
    .groupBy("message_key")
    .agg(
        F.count("*").alias("record_count"),
        F.countDistinct(
            F.struct(*[F.col(c) for c in business_key])
        ).alias("distinct_trips")
    )
)

key_mapping.select(
    F.count("*").alias("distinct_keys"),
    F.sum(
        F.when(F.col("record_count") == 3, 1).otherwise(0)
    ).alias("keys_appearing_3_times"),
    F.sum(
        F.when(F.col("distinct_trips") == 1, 1).otherwise(0)
    ).alias("keys_mapping_to_one_trip"),
    F.sum(
        F.when(F.col("distinct_trips") > 1, 1).otherwise(0)
    ).alias("keys_mapping_to_multiple_trips")
).show()

In [0]:
phase4_idempotency_table = "main.`nyc-taxi`.phase4_idempotency_test"

if spark.catalog.tableExists(phase4_idempotency_table):
    print("STOP: Idempotency test table already exists.")
else:
    print("READY: Idempotency test table does not exist.")

In [0]:
(
    valid_df
    .limit(3)
    .write
    .format("delta")
    .mode("errorifexists")
    .saveAsTable(phase4_idempotency_table)
)

print("Created isolated idempotency test table.")
print(
    "Initial row count:",
    spark.table(phase4_idempotency_table).count()
)

In [0]:
if spark.catalog.tableExists(phase4_idempotency_table):
    print("STOP: Test table already exists. No data written.")
else:
    (
        valid_df
        .limit(3)
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(phase4_idempotency_table)
    )

    print("Created isolated idempotency test table.")
    print(
        "Initial row count:",
        spark.table(phase4_idempotency_table).count()
    )

In [0]:
(
    valid_df
    .limit(3)
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(phase4_idempotency_table)
)

print(
    "Row count after second append:",
    spark.table(phase4_idempotency_table).count()
)

In [0]:
phase4_merge_test_table = (
    "main.`nyc-taxi`.phase4_idempotency_merge_test"
)

df_merge_sample = (
    valid_df
    .orderBy("kafka_partition", "kafka_offset")
    .limit(3)
)

if spark.catalog.tableExists(phase4_merge_test_table):
    print("STOP: MERGE test table already exists. No data written.")
else:
    (
        df_merge_sample.write
        .format("delta")
        .mode("append")
        .saveAsTable(phase4_merge_test_table)
    )

    print("Created MERGE test table.")
    print(
        "Initial row count:",
        spark.table(phase4_merge_test_table).count()
    )

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    phase4_merge_test_table
)

(
    target.alias("target")
    .merge(
        df_merge_sample.alias("source"),
        """
        target.topic = source.topic
        AND target.kafka_partition = source.kafka_partition
        AND target.kafka_offset = source.kafka_offset
        """
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print(
    "Row count after MERGE:",
    spark.table(phase4_merge_test_table).count()
)

In [0]:
target = DeltaTable.forName(
    spark,
    phase4_merge_test_table
)

(
    target.alias("target")
    .merge(
        df_phase4_valid.alias("source"),
        """
        target.topic = source.topic
        AND target.kafka_partition = source.kafka_partition
        AND target.kafka_offset = source.kafka_offset
        """
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print(
    "MERGE test table count:",
    spark.table(phase4_merge_test_table).count()
)

In [0]:
target = DeltaTable.forName(
    spark,
    phase4_merge_test_table
)

(
    target.alias("target")
    .merge(
        df_phase4_valid.alias("source"),
        """
        target.topic = source.topic
        AND target.kafka_partition = source.kafka_partition
        AND target.kafka_offset = source.kafka_offset
        """
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print(
    "Row count after repeated MERGE:",
    spark.table(phase4_merge_test_table).count()
)

print(
    "Distinct Kafka identities:",
    spark.table(phase4_merge_test_table)
    .select("topic", "kafka_partition", "kafka_offset")
    .distinct()
    .count()
)

In [0]:
merge_df = spark.table(phase4_merge_test_table)

key_cols = ["topic", "kafka_partition", "kafka_offset"]

merge_count = merge_df.count()
merge_unique_count = merge_df.select(*key_cols).distinct().count()

valid_keys = valid_df.select(*key_cols).distinct()
merge_keys = merge_df.select(*key_cols).distinct()

missing_from_merge = valid_keys.join(
    merge_keys, key_cols, "left_anti"
).count()

unexpected_in_merge = merge_keys.join(
    valid_keys, key_cols, "left_anti"
).count()

print("MERGE table rows:", merge_count)
print("Unique Kafka identities:", merge_unique_count)
print("Valid identities missing from MERGE table:", missing_from_merge)
print("Unexpected identities in MERGE table:", unexpected_in_merge)

In [0]:
checkpoint_root = (
    "/Volumes/main/nyc-taxi/nyc_taxi_data/checkpoints"
)

display(
    dbutils.fs.ls(checkpoint_root)
)

In [0]:
phase4_recovery_checkpoint = (
    "/Volumes/main/nyc-taxi/nyc_taxi_data/"
    "checkpoints/phase4_recovery_test"
)

try:
    checkpoint_contents = dbutils.fs.ls(
        phase4_recovery_checkpoint
    )
    print("STOP: Recovery checkpoint path already exists.")
    print(checkpoint_contents)
except Exception as e:
    error_text = str(e).lower()

    if (
        "not found" in error_text
        or "does not exist" in error_text
        or "filenotfound" in error_text
    ):
        print("READY: Dedicated recovery checkpoint path is absent.")
    else:
        raise

In [0]:
phase4_recovery_table = (
    "main.`nyc-taxi`.phase4_recovery_test"
)

if spark.catalog.tableExists(phase4_recovery_table):
    print("STOP: Recovery test table already exists.")
else:
    print("READY: Recovery test table does not exist.")

In [0]:
phase4_recovery_table = (
    "main.`nyc-taxi`.phase4_recovery_test"
)

if spark.catalog.tableExists(phase4_recovery_table):
    print("STOP: Recovery test table already exists.")
else:
    print("READY: Recovery test table does not exist.")

In [0]:
raw_saved_df = spark.table(
    "main.`nyc-taxi`.phase4_raw_kafka_messages"
)

print("Captured messages:", raw_saved_df.count())

print(
    "Distinct Kafka identities:",
    raw_saved_df
    .select("topic", "kafka_partition", "kafka_offset")
    .distinct()
    .count()
)

In [0]:
from pyspark.sql import functions as F

raw_saved_df = spark.table(
    "main.`nyc-taxi`.phase4_raw_kafka_messages"
)

replayed_df = raw_saved_df.withColumn(
    "taxi",
    F.from_json(
        F.col("raw_value"),
        taxi_schema
    )
)

replayed_df = replayed_df.withColumn(
    "validation_status",
    F.when(
        F.col("taxi.VendorID").isNull(),
        F.lit("QUARANTINE_MISSING_REQUIRED_FIELDS")
    ).otherwise(F.lit("VALID"))
)

replayed_df.groupBy("validation_status").count().show()

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    LongType, DoubleType, StringType,
    IntegerType, TimestampNTZType
)

taxi_schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampNTZType(), True),
    StructField("tpep_dropoff_datetime", TimestampNTZType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", IntegerType(), True),
    StructField("airport_fee", IntegerType(), True)
])

print("Taxi schema recreated successfully.")

In [0]:
from pyspark.sql import functions as F

raw_saved_df = spark.table(
    "main.`nyc-taxi`.phase4_raw_kafka_messages"
)

replayed_df = raw_saved_df.withColumn(
    "taxi",
    F.from_json(F.col("raw_value"), taxi_schema)
)

replayed_df = replayed_df.withColumn(
    "validation_status",
    F.when(
        F.col("taxi.VendorID").isNull(),
        F.lit("QUARANTINE_MISSING_REQUIRED_FIELDS")
    ).otherwise(F.lit("VALID"))
)

replayed_df.groupBy("validation_status").count().show()

In [0]:
key_cols = ["topic", "kafka_partition", "kafka_offset"]

replayed_valid_keys = (
    replayed_df
    .filter(F.col("validation_status") == "VALID")
    .select(*key_cols)
    .distinct()
)

replayed_quarantine_keys = (
    replayed_df
    .filter(F.col("validation_status") != "VALID")
    .select(*key_cols)
    .distinct()
)

saved_valid_keys = (
    spark.table("main.`nyc-taxi`.phase4_valid_taxi_events")
    .select(*key_cols)
    .distinct()
)

saved_quarantine_keys = (
    spark.table("main.`nyc-taxi`.phase4_quarantine_taxi_events")
    .select(*key_cols)
    .distinct()
)

print(
    "Valid identity differences:",
    replayed_valid_keys.join(
        saved_valid_keys, key_cols, "left_anti"
    ).count()
    + saved_valid_keys.join(
        replayed_valid_keys, key_cols, "left_anti"
    ).count()
)

print(
    "Quarantine identity differences:",
    replayed_quarantine_keys.join(
        saved_quarantine_keys, key_cols, "left_anti"
    ).count()
    + saved_quarantine_keys.join(
        replayed_quarantine_keys, key_cols, "left_anti"
    ).count()
)


In [0]:

from pyspark.sql import functions as F

# Read existing retained Kafka messages in batch mode only
df_kafka_inspection = (
    spark.read
    .format("kafka")
    .options(**kafka_options)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load()
)

# Summarize the messages currently available in the topic
df_kafka_offsets = (
    df_kafka_inspection
    .groupBy("topic", "partition")
    .agg(
        F.count("*").alias("message_count"),
        F.min("offset").alias("first_offset"),
        F.max("offset").alias("last_offset")
    )
    .orderBy("partition")
)

display(df_kafka_offsets)


In [0]:

# New resources reserved exclusively for the recovery test
recovery_checkpoint = (
    "/Volumes/main/nyc-taxi/nyc_taxi_data/"
    "checkpoints/phase4_recovery_test"
)
recovery_table = "main.`nyc-taxi`.phase4_recovery_test"

# Check whether the checkpoint path already exists
checkpoint_exists = any(
    item.name.rstrip("/") == "phase4_recovery_test"
    for item in dbutils.fs.ls(
        "/Volumes/main/nyc-taxi/nyc_taxi_data/checkpoints"
    )
)

# Check whether the destination table already exists
table_exists = spark.catalog.tableExists(recovery_table)

print("Checkpoint already exists:", checkpoint_exists)
print("Destination table already exists:", table_exists)

if not checkpoint_exists and not table_exists:
    print("READY: Both recovery-test resources are unused.")
else:
    print("STOP: At least one resource already exists. Do not start the test.")


In [0]:

from pyspark.sql import functions as F

# Read the existing Kafka topic as a streaming source
df_recovery_stream = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
    .select(
        F.col("topic"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
        F.col("timestamp").alias("kafka_timestamp"),
        F.col("key").cast("string").alias("message_key"),
        F.col("value").cast("string").alias("raw_value"),
        F.current_timestamp().alias("captured_at")
    )
)

recovery_query = (
    df_recovery_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", recovery_checkpoint)
    .trigger(availableNow=True)
    .toTable(recovery_table)
)

recovery_query.awaitTermination()

print("First recovery-test pass completed.")


In [0]:

from pyspark.sql import functions as F

df_recovery_result = spark.table(recovery_table)

total_rows = df_recovery_result.count()

unique_kafka_messages = (
    df_recovery_result
    .select("topic", "kafka_partition", "kafka_offset")
    .distinct()
    .count()
)

duplicate_kafka_messages = total_rows - unique_kafka_messages

print("Total rows:", total_rows)
print("Unique Kafka messages:", unique_kafka_messages)
print("Duplicate Kafka identities:", duplicate_kafka_messages)

display(
    df_recovery_result
    .groupBy("kafka_partition")
    .agg(
        F.count("*").alias("message_count"),
        F.min("kafka_offset").alias("first_offset"),
        F.max("kafka_offset").alias("last_offset")
    )
    .orderBy("kafka_partition")
)


In [0]:

df_recovery_stream_restart = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
    .select(
        F.col("topic"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
        F.col("timestamp").alias("kafka_timestamp"),
        F.col("key").cast("string").alias("message_key"),
        F.col("value").cast("string").alias("raw_value"),
        F.current_timestamp().alias("captured_at")
    )
)

recovery_query_restart = (
    df_recovery_stream_restart.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", recovery_checkpoint)
    .trigger(availableNow=True)
    .toTable(recovery_table)
)

recovery_query_restart.awaitTermination()

print("Restart pass completed using the existing checkpoint.")


In [0]:

df_recovery_after_restart = spark.table(recovery_table)

rows_after_restart = df_recovery_after_restart.count()

unique_messages_after_restart = (
    df_recovery_after_restart
    .select("topic", "kafka_partition", "kafka_offset")
    .distinct()
    .count()
)

duplicates_after_restart = (
    rows_after_restart - unique_messages_after_restart
)

print("Rows before restart:", 302)
print("Rows after restart:", rows_after_restart)
print("Unique Kafka identities:", unique_messages_after_restart)
print("Duplicate Kafka identities:", duplicates_after_restart)

if rows_after_restart == 302 and duplicates_after_restart == 0:
    print("PASS: No duplicate rows appeared after the checkpoint restart.")
else:
    print("CHECK REQUIRED: The result differs from the expected baseline.")


In [0]:

recovery_table = "main.`nyc-taxi`.phase4_recovery_test"

print("Recovery table schema:")
spark.table(recovery_table).printSchema()

print("Recovery table row count:")
print(spark.table(recovery_table).count())

print("Sample captured messages:")
display(
    spark.table(recovery_table)
    .select(
        "topic",
        "kafka_partition",
        "kafka_offset",
        "raw_value"
    )
    .orderBy("kafka_partition", "kafka_offset")
    .limit(5)
)


In [0]:

from pyspark.sql import functions as F

from pyspark.sql.types import (
    StructType, StructField,
    LongType, DoubleType, StringType,
    IntegerType, TimestampNTZType
)

taxi_schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampNTZType(), True),
    StructField("tpep_dropoff_datetime", TimestampNTZType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", IntegerType(), True),
    StructField("airport_fee", IntegerType(), True)
])

df_recovery_parsed = (
    spark.table(recovery_table)
    .withColumn("taxi", F.from_json(F.col("raw_value"), taxi_schema))
    .withColumn(
        "validation_status",
        F.when(
            F.col("taxi").isNull(),
            F.lit("QUARANTINE_INVALID_JSON")
        ).when(
            F.col("taxi.VendorID").isNull()
            | F.col("taxi.tpep_pickup_datetime").isNull()
            | F.col("taxi.tpep_dropoff_datetime").isNull()
            | F.col("taxi.passenger_count").isNull()
            | F.col("taxi.trip_distance").isNull()
            | F.col("taxi.total_amount").isNull(),
            F.lit("QUARANTINE_MISSING_REQUIRED_FIELDS")
        ).otherwise(F.lit("VALID"))
    )
)

display(
    df_recovery_parsed
    .groupBy("validation_status")
    .count()
    .orderBy("validation_status")
)


In [0]:
display(
    df_recovery_parsed
    .filter(F.col("validation_status") != "VALID")
    .select(
        "topic",
        "kafka_partition",
        "kafka_offset",
        "raw_value",
        "validation_status"
    )
    .orderBy("kafka_partition", "kafka_offset")
)

In [0]:
# New resources for the Phase 4 streaming pipeline
streaming_raw_table = "main.`nyc-taxi`.phase4_streaming_raw"
streaming_valid_table = "main.`nyc-taxi`.phase4_streaming_valid"
streaming_quarantine_table = "main.`nyc-taxi`.phase4_streaming_quarantine"

streaming_checkpoint = (
    "/Volumes/main/nyc-taxi/nyc_taxi_data/checkpoints/"
    "phase4_streaming_pipeline"
)

print("New streaming pipeline resources:")
print("Raw table:", streaming_raw_table)
print("Valid table:", streaming_valid_table)
print("Quarantine table:", streaming_quarantine_table)
print("Checkpoint:", streaming_checkpoint)

In [0]:
# Check whether the three target tables already exist
target_tables = [
    streaming_raw_table,
    streaming_valid_table,
    streaming_quarantine_table
]

for table_name in target_tables:
    exists = spark.catalog.tableExists(table_name)
    print(f"{table_name}: {'EXISTS' if exists else 'NOT FOUND'}")

# Check whether the checkpoint directory already exists
checkpoint_exists = False

try:
    checkpoint_exists = any(
        item.path.rstrip("/") == streaming_checkpoint.rstrip("/")
        for item in dbutils.fs.ls(
            "/Volumes/main/nyc-taxi/nyc_taxi_data/checkpoints/"
        )
    )
except Exception as e:
    print("Could not inspect checkpoint directory:", str(e))

print(
    "Checkpoint:",
    "EXISTS — do not reuse without inspection"
    if checkpoint_exists
    else "NOT FOUND at the inspected path"
)

In [0]:
from pyspark.sql import functions as F

df_phase4_raw_stream = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
    .select(
        F.col("topic"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
        F.col("timestamp").alias("kafka_timestamp"),
        F.col("key").cast("string").alias("message_key"),
        F.col("value").cast("string").alias("raw_value"),
        F.current_timestamp().alias("captured_at")
    )
)

print("Raw streaming DataFrame defined.")
print("Streaming source:", KAFKA_TOPIC)
print("Target table:", streaming_raw_table)
print("Checkpoint:", streaming_checkpoint)

In [0]:
df_phase4_raw_stream.printSchema()

In [0]:
raw_stream_query = (
    df_phase4_raw_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", streaming_checkpoint)
    .trigger(availableNow=True)
    .toTable(streaming_raw_table)
)

raw_stream_query.awaitTermination()

print("Raw Kafka ingestion completed.")

In [0]:
from pyspark.sql import functions as F

df_raw_check = spark.table(streaming_raw_table)

total_rows = df_raw_check.count()

unique_kafka_ids = (
    df_raw_check
    .select("topic", "kafka_partition", "kafka_offset")
    .distinct()
    .count()
)

print("Total raw messages:", total_rows)
print("Unique Kafka identities:", unique_kafka_ids)
print("Duplicate identities:", total_rows - unique_kafka_ids)

display(
    df_raw_check
    .groupBy("kafka_partition")
    .agg(
        F.count("*").alias("message_count"),
        F.min("kafka_offset").alias("first_offset"),
        F.max("kafka_offset").alias("last_offset")
    )
    .orderBy("kafka_partition")
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    LongType, DoubleType, StringType,
    IntegerType, TimestampNTZType
)

taxi_schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampNTZType(), True),
    StructField("tpep_dropoff_datetime", TimestampNTZType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", IntegerType(), True),
    StructField("airport_fee", IntegerType(), True)
])

df_phase4_classified = (
    spark.table(streaming_raw_table)
    .withColumn("taxi", F.from_json(F.col("raw_value"), taxi_schema))
    .withColumn(
        "validation_status",
        F.when(
            F.col("taxi").isNull(),
            F.lit("QUARANTINE_INVALID_JSON")
        ).when(
            F.col("taxi.VendorID").isNull()
            | F.col("taxi.tpep_pickup_datetime").isNull()
            | F.col("taxi.tpep_dropoff_datetime").isNull()
            | F.col("taxi.passenger_count").isNull()
            | F.col("taxi.trip_distance").isNull()
            | F.col("taxi.total_amount").isNull(),
            F.lit("QUARANTINE_MISSING_REQUIRED_FIELDS")
        ).otherwise(F.lit("VALID"))
    )
)

display(
    df_phase4_classified
    .groupBy("validation_status")
    .count()
    .orderBy("validation_status")
)

In [0]:
df_phase4_valid = (
    df_phase4_classified
    .filter(F.col("validation_status") == "VALID")
    .select(
        "topic",
        "kafka_partition",
        "kafka_offset",
        "kafka_timestamp",
        "message_key",
        "raw_value",
        "captured_at",
        "taxi.*"
    )
)

df_phase4_quarantine = (
    df_phase4_classified
    .filter(F.col("validation_status") != "VALID")
    .select(
        "topic",
        "kafka_partition",
        "kafka_offset",
        "kafka_timestamp",
        "message_key",
        "raw_value",
        "captured_at",
        "validation_status"
    )
)

print("Valid DataFrame:", df_phase4_valid.count(), "rows")
print("Quarantine DataFrame:", df_phase4_quarantine.count(), "rows")

In [0]:
print("=== VALID OUTPUT SCHEMA ===")
df_phase4_valid.printSchema()

print("=== QUARANTINE OUTPUT SCHEMA ===")
df_phase4_quarantine.printSchema()

In [0]:
for table_name in [streaming_valid_table, streaming_quarantine_table]:
    exists = spark.catalog.tableExists(table_name)
    print(f"{table_name}: {'EXISTS' if exists else 'NOT FOUND'}")

    if exists:
        print("Rows:", spark.table(table_name).count())

In [0]:
for table_name, output_df in [
    (streaming_valid_table, df_phase4_valid),
    (streaming_quarantine_table, df_phase4_quarantine)
]:
    if spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Safety stop: {table_name} already exists. "
            "No overwrite performed."
        )

    (
        output_df.limit(0)
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )

    print("Created empty Delta table:", table_name)

In [0]:
for table_name in [streaming_valid_table, streaming_quarantine_table]:
    count = spark.table(table_name).count()
    print(f"{table_name}: {count} rows")

In [0]:
from delta.tables import DeltaTable

valid_delta = DeltaTable.forName(spark, streaming_valid_table)

(
    valid_delta.alias("target")
    .merge(
        df_phase4_valid.alias("source"),
        """
        target.topic = source.topic
        AND target.kafka_partition = source.kafka_partition
        AND target.kafka_offset = source.kafka_offset
        """
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Valid-event MERGE completed.")

In [0]:
df_valid_check = spark.table(streaming_valid_table)

valid_count = df_valid_check.count()
valid_unique_ids = (
    df_valid_check
    .select("topic", "kafka_partition", "kafka_offset")
    .distinct()
    .count()
)

print("Valid rows:", valid_count)
print("Unique Kafka identities:", valid_unique_ids)
print("Duplicate identities:", valid_count - valid_unique_ids)

In [0]:
from delta.tables import DeltaTable

quarantine_delta = DeltaTable.forName(
    spark, streaming_quarantine_table
)

(
    quarantine_delta.alias("target")
    .merge(
        df_phase4_quarantine.alias("source"),
        """
        target.topic = source.topic
        AND target.kafka_partition = source.kafka_partition
        AND target.kafka_offset = source.kafka_offset
        """
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Quarantine MERGE completed.")

In [0]:
raw_count = spark.table(streaming_raw_table).count()

valid_df = spark.table(streaming_valid_table)
quarantine_df = spark.table(streaming_quarantine_table)

valid_count = valid_df.count()
quarantine_count = quarantine_df.count()

valid_ids = valid_df.select(
    "topic", "kafka_partition", "kafka_offset"
).distinct()

quarantine_ids = quarantine_df.select(
    "topic", "kafka_partition", "kafka_offset"
).distinct()

overlap_count = (
    valid_ids.join(
        quarantine_ids,
        ["topic", "kafka_partition", "kafka_offset"],
        "inner"
    ).count()
)

output_ids = valid_ids.unionByName(quarantine_ids).distinct()
raw_ids = (
    spark.table(streaming_raw_table)
    .select("topic", "kafka_partition", "kafka_offset")
    .distinct()
)

missing_count = raw_ids.join(
    output_ids,
    ["topic", "kafka_partition", "kafka_offset"],
    "left_anti"
).count()

unexpected_count = output_ids.join(
    raw_ids,
    ["topic", "kafka_partition", "kafka_offset"],
    "left_anti"
).count()

print("Raw rows:", raw_count)
print("Valid rows:", valid_count)
print("Quarantine rows:", quarantine_count)
print("Total output rows:", valid_count + quarantine_count)
print("Overlap between valid and quarantine:", overlap_count)
print("Raw events missing from outputs:", missing_count)
print("Unexpected output identities:", unexpected_count)

assert raw_count == valid_count + quarantine_count
assert overlap_count == 0
assert missing_count == 0
assert unexpected_count == 0

print("PASS: All raw events are reconciled.")

In [0]:
from delta.tables import DeltaTable

# Repeat the valid-event MERGE
(
    DeltaTable.forName(spark, streaming_valid_table).alias("target")
    .merge(
        df_phase4_valid.alias("source"),
        """
        target.topic = source.topic
        AND target.kafka_partition = source.kafka_partition
        AND target.kafka_offset = source.kafka_offset
        """
    )
    .whenNotMatchedInsertAll()
    .execute()
)

# Repeat the quarantine MERGE
(
    DeltaTable.forName(spark, streaming_quarantine_table).alias("target")
    .merge(
        df_phase4_quarantine.alias("source"),
        """
        target.topic = source.topic
        AND target.kafka_partition = source.kafka_partition
        AND target.kafka_offset = source.kafka_offset
        """
    )
    .whenNotMatchedInsertAll()
    .execute()
)

valid_after_retry = spark.table(streaming_valid_table).count()
quarantine_after_retry = spark.table(streaming_quarantine_table).count()

print("Valid rows after repeated MERGE:", valid_after_retry)
print("Quarantine rows after repeated MERGE:", quarantine_after_retry)

assert valid_after_retry == 300
assert quarantine_after_retry == 2

print("PASS: Repeated MERGE did not add duplicate rows.")

In [0]:
downstream_checkpoints = {
    "valid": (
        "/Volumes/main/nyc-taxi/nyc_taxi_data/checkpoints/"
        "phase4_valid_from_raw"
    ),
    "quarantine": (
        "/Volumes/main/nyc-taxi/nyc_taxi_data/checkpoints/"
        "phase4_quarantine_from_raw"
    )
}

checkpoint_root = (
    "/Volumes/main/nyc-taxi/nyc_taxi_data/checkpoints/"
)

existing_paths = {
    item.path.rstrip("/")
    for item in dbutils.fs.ls(checkpoint_root)
}

for name, path in downstream_checkpoints.items():
    exists = path.rstrip("/") in existing_paths
    print(f"{name}: {path} -> {'EXISTS' if exists else 'NOT FOUND'}")

In [0]:
df_raw_delta_stream = (
    spark.readStream
    .format("delta")
    .table(streaming_raw_table)
)

print("Downstream source defined:", streaming_raw_table)
df_raw_delta_stream.printSchema()

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

def merge_valid_batch(batch_df, batch_id):
    parsed_batch = (
        batch_df
        .withColumn("taxi", F.from_json(F.col("raw_value"), taxi_schema))
        .filter(
            F.col("taxi.VendorID").isNotNull()
            & F.col("taxi.tpep_pickup_datetime").isNotNull()
            & F.col("taxi.tpep_dropoff_datetime").isNotNull()
            & F.col("taxi.passenger_count").isNotNull()
            & F.col("taxi.trip_distance").isNotNull()
            & F.col("taxi.total_amount").isNotNull()
        )
        .select(
            "topic",
            "kafka_partition",
            "kafka_offset",
            "kafka_timestamp",
            "message_key",
            "raw_value",
            "captured_at",
            "taxi.*"
        )
    )

    target = DeltaTable.forName(spark, streaming_valid_table)

    (
        target.alias("target")
        .merge(
            parsed_batch.alias("source"),
            """
            target.topic = source.topic
            AND target.kafka_partition = source.kafka_partition
            AND target.kafka_offset = source.kafka_offset
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Valid batch {batch_id} merged.")

print("Valid-event foreachBatch function defined.")

In [0]:
def merge_quarantine_batch(batch_df, batch_id):
    parsed_batch = (
        batch_df
        .withColumn("taxi", F.from_json(F.col("raw_value"), taxi_schema))
        .withColumn(
            "validation_status",
            F.when(
                F.col("taxi").isNull(),
                F.lit("QUARANTINE_INVALID_JSON")
            ).when(
                F.col("taxi.VendorID").isNull()
                | F.col("taxi.tpep_pickup_datetime").isNull()
                | F.col("taxi.tpep_dropoff_datetime").isNull()
                | F.col("taxi.passenger_count").isNull()
                | F.col("taxi.trip_distance").isNull()
                | F.col("taxi.total_amount").isNull(),
                F.lit("QUARANTINE_MISSING_REQUIRED_FIELDS")
            )
        )
        .filter(F.col("validation_status").isNotNull())
        .select(
            "topic",
            "kafka_partition",
            "kafka_offset",
            "kafka_timestamp",
            "message_key",
            "raw_value",
            "captured_at",
            "validation_status"
        )
    )

    target = DeltaTable.forName(spark, streaming_quarantine_table)

    (
        target.alias("target")
        .merge(
            parsed_batch.alias("source"),
            """
            target.topic = source.topic
            AND target.kafka_partition = source.kafka_partition
            AND target.kafka_offset = source.kafka_offset
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Quarantine batch {batch_id} merged.")

print("Quarantine foreachBatch function defined.")

In [0]:
valid_stream_query = (
    df_raw_delta_stream.writeStream
    .foreachBatch(merge_valid_batch)
    .option("checkpointLocation", downstream_checkpoints["valid"])
    .trigger(availableNow=True)
    .start()
)

valid_stream_query.awaitTermination()

print("Valid downstream stream completed.")

In [0]:
df_valid_stream_check = spark.table(streaming_valid_table)

valid_rows = df_valid_stream_check.count()
valid_unique_ids = (
    df_valid_stream_check
    .select("topic", "kafka_partition", "kafka_offset")
    .distinct()
    .count()
)

print("Valid rows:", valid_rows)
print("Unique Kafka identities:", valid_unique_ids)
print("Duplicate identities:", valid_rows - valid_unique_ids)

assert valid_rows == 300
assert valid_unique_ids == 300

print("PASS: Valid downstream stream verified.")

In [0]:
quarantine_stream_query = (
    df_raw_delta_stream.writeStream
    .foreachBatch(merge_quarantine_batch)
    .option("checkpointLocation", downstream_checkpoints["quarantine"])
    .trigger(availableNow=True)
    .start()
)

quarantine_stream_query.awaitTermination()

print("Quarantine downstream stream completed.")

In [0]:
raw_df = spark.table(streaming_raw_table)
valid_df = spark.table(streaming_valid_table)
quarantine_df = spark.table(streaming_quarantine_table)

raw_count = raw_df.count()
valid_count = valid_df.count()
quarantine_count = quarantine_df.count()

valid_ids = valid_df.select(
    "topic", "kafka_partition", "kafka_offset"
).distinct()

quarantine_ids = quarantine_df.select(
    "topic", "kafka_partition", "kafka_offset"
).distinct()

raw_ids = raw_df.select(
    "topic", "kafka_partition", "kafka_offset"
).distinct()

overlap = valid_ids.join(
    quarantine_ids,
    ["topic", "kafka_partition", "kafka_offset"],
    "inner"
).count()

outputs = valid_ids.unionByName(quarantine_ids).distinct()

missing = raw_ids.join(
    outputs,
    ["topic", "kafka_partition", "kafka_offset"],
    "left_anti"
).count()

unexpected = outputs.join(
    raw_ids,
    ["topic", "kafka_partition", "kafka_offset"],
    "left_anti"
).count()

print("Raw rows:", raw_count)
print("Valid rows:", valid_count)
print("Quarantine rows:", quarantine_count)
print("Total output rows:", valid_count + quarantine_count)
print("Duplicate valid identities:", valid_count - valid_ids.count())
print("Duplicate quarantine identities:", quarantine_count - quarantine_ids.count())
print("Valid/quarantine overlap:", overlap)
print("Missing raw identities:", missing)
print("Unexpected output identities:", unexpected)

assert raw_count == 302
assert valid_count == 300
assert quarantine_count == 2
assert valid_count + quarantine_count == raw_count
assert valid_count == valid_ids.count()
assert quarantine_count == quarantine_ids.count()
assert overlap == 0
assert missing == 0
assert unexpected == 0

print("PASS: Final downstream streaming verification.")

In [0]:
for name, query in [
    ("VALID", valid_stream_query),
    ("QUARANTINE", quarantine_stream_query)
]:
    print(f"--- {name} STREAM ---")
    print("Query ID:", query.id)
    print("Is active:", query.isActive)
    print("Status:", query.status)
    print("Last progress:", query.lastProgress)

In [0]:
df_raw_now = spark.table(streaming_raw_table)

print("Current raw row count:", df_raw_now.count())

display(
    df_raw_now
    .groupBy("kafka_partition")
    .agg(
        F.count("*").alias("message_count"),
        F.max("kafka_offset").alias("latest_offset")
    )
    .orderBy("kafka_partition")
)

In [0]:
valid_restart_query = (
    spark.readStream
    .format("delta")
    .table(streaming_raw_table)
    .writeStream
    .foreachBatch(merge_valid_batch)
    .option("checkpointLocation", downstream_checkpoints["valid"])
    .trigger(availableNow=True)
    .start()
)

valid_restart_query.awaitTermination()

print("Valid stream checkpoint restart completed.")
print("Query status:", valid_restart_query.status)
print("Last progress:", valid_restart_query.lastProgress)

In [0]:
quarantine_restart_query = (
    spark.readStream
    .format("delta")
    .table(streaming_raw_table)
    .writeStream
    .foreachBatch(merge_quarantine_batch)
    .option("checkpointLocation", downstream_checkpoints["quarantine"])
    .trigger(availableNow=True)
    .start()
)

quarantine_restart_query.awaitTermination()

print("Quarantine stream checkpoint restart completed.")
print("Query status:", quarantine_restart_query.status)
print("Last progress:", quarantine_restart_query.lastProgress)

In [0]:

# Isolated incremental-processing test resources
test_source_table = "main.`nyc-taxi`.phase4_incremental_test_source"
test_valid_table = "main.`nyc-taxi`.phase4_incremental_test_valid"
test_quarantine_table = "main.`nyc-taxi`.phase4_incremental_test_quarantine"

test_checkpoints = {
    "valid": "/Volumes/main/nyc-taxi/nyc_taxi_data/checkpoints/phase4_incremental_test_valid",
    "quarantine": "/Volumes/main/nyc-taxi/nyc_taxi_data/checkpoints/phase4_incremental_test_quarantine"
}

# Check whether any of the proposed tables already exist.
for table_name in [
    test_source_table,
    test_valid_table,
    test_quarantine_table
]:
    exists = spark.catalog.tableExists(table_name)
    print(f"{table_name}: {'EXISTS' if exists else 'NOT FOUND'}")

# Check the proposed checkpoint directories without modifying them.
for label, path in test_checkpoints.items():
    try:
        dbutils.fs.ls(path)
        print(f"Checkpoint {label}: EXISTS")
    except Exception as e:
        message = str(e).lower()
        if "filenotfound" in message or "does not exist" in message or "path_not_found" in message:
            print(f"Checkpoint {label}: NOT FOUND")
        else:
            print(f"Checkpoint {label}: CHECK FAILED — {e}")


In [0]:

# Safety check: stop if any test table now exists.
test_tables = [
    test_source_table,
    test_valid_table,
    test_quarantine_table
]

existing_tables = [
    name for name in test_tables
    if spark.catalog.tableExists(name)
]

if existing_tables:
    raise RuntimeError(
        f"Safety stop: these test tables already exist: {existing_tables}"
    )

# Create an empty source with the same schema as the raw streaming table.
(
    spark.table(streaming_raw_table)
    .limit(0)
    .write.format("delta")
    .saveAsTable(test_source_table)
)

# Create empty output tables using the existing output schemas.
(
    spark.table(streaming_valid_table)
    .limit(0)
    .write.format("delta")
    .saveAsTable(test_valid_table)
)

(
    spark.table(streaming_quarantine_table)
    .limit(0)
    .write.format("delta")
    .saveAsTable(test_quarantine_table)
)

# Verify that all three tables are empty.
for table_name in test_tables:
    print(f"{table_name}: {spark.table(table_name).count()} rows")

print("Isolated test tables created and verified.")


In [0]:

# Restore Phase 4 table names
streaming_raw_table = "main.`nyc-taxi`.phase4_streaming_raw"
streaming_valid_table = "main.`nyc-taxi`.phase4_streaming_valid"
streaming_quarantine_table = "main.`nyc-taxi`.phase4_streaming_quarantine"

# Define isolated test tables
test_source_table = "main.`nyc-taxi`.phase4_incremental_test_source"
test_valid_table = "main.`nyc-taxi`.phase4_incremental_test_valid"
test_quarantine_table = "main.`nyc-taxi`.phase4_incremental_test_quarantine"

# Safety check: do not overwrite anything
test_tables = [
    test_source_table,
    test_valid_table,
    test_quarantine_table
]

for table_name in test_tables:
    exists = spark.catalog.tableExists(table_name)
    print(f"{table_name}: {'EXISTS' if exists else 'NOT FOUND'}")

# Confirm the existing raw table is available
if not spark.catalog.tableExists(streaming_raw_table):
    raise RuntimeError(
        f"Expected existing raw table not found: {streaming_raw_table}"
    )

print("Existing Phase 4 raw table is available.")


In [0]:
# Safety check: never overwrite an existing test table.
test_tables = [
    test_source_table,
    test_valid_table,
    test_quarantine_table
]

existing_tables = [
    name for name in test_tables
    if spark.catalog.tableExists(name)
]

if existing_tables:
    raise RuntimeError(
        f"Safety stop — existing tables found: {existing_tables}"
    )

# Create empty tables using the schemas of the existing Phase 4 tables.
(
    spark.table(streaming_raw_table)
    .limit(0)
    .write.format("delta")
    .saveAsTable(test_source_table)
)

(
    spark.table(streaming_valid_table)
    .limit(0)
    .write.format("delta")
    .saveAsTable(test_valid_table)
)

(
    spark.table(streaming_quarantine_table)
    .limit(0)
    .write.format("delta")
    .saveAsTable(test_quarantine_table)
)

# Verify
for table_name in test_tables:
    print(f"{table_name}: {spark.table(table_name).count()} rows")

print("PASS: Three isolated, empty Delta tables created.")

In [0]:
from pyspark.sql import functions as F

# Safety checks
if spark.table(test_source_table).count() != 0:
    raise RuntimeError("Safety stop: test source is not empty.")

if spark.table(test_valid_table).count() != 0:
    raise RuntimeError("Safety stop: test valid table is not empty.")

if spark.table(test_quarantine_table).count() != 0:
    raise RuntimeError("Safety stop: test quarantine table is not empty.")

# Select one existing valid raw message as a template.
template = (
    spark.table(streaming_raw_table)
    .filter(F.col("raw_value").isNotNull())
    .filter(F.from_json(F.col("raw_value"), taxi_schema).isNotNull())
    .filter(F.from_json(F.col("raw_value"), taxi_schema)["VendorID"].isNotNull())
    .select("raw_value", "kafka_timestamp", "captured_at")
    .limit(1)
)

if template.count() != 1:
    raise RuntimeError("Could not find exactly one valid template event.")

# Give the synthetic event a separate, clearly marked identity.
synthetic_event = template.select(
    F.lit("taxi-trips-incremental-test").alias("topic"),
    F.lit(999).cast("int").alias("kafka_partition"),
    F.lit(0).cast("long").alias("kafka_offset"),
    F.col("kafka_timestamp"),
    F.lit("synthetic-incremental-test").alias("message_key"),
    F.col("raw_value"),
    F.current_timestamp().alias("captured_at")
)

# Append only to the isolated test source.
synthetic_event.write.format("delta").mode("append").saveAsTable(
    test_source_table
)

print("Synthetic test event inserted into the isolated source.")
print("Test source rows:", spark.table(test_source_table).count())
print("Production raw rows:", spark.table(streaming_raw_table).count())

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    LongType, TimestampNTZType,
    DoubleType, StringType, IntegerType
)

taxi_schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampNTZType(), True),
    StructField("tpep_dropoff_datetime", TimestampNTZType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", IntegerType(), True),
    StructField("airport_fee", IntegerType(), True)
])

print("Taxi JSON schema restored.")
print("Field count:", len(taxi_schema.fields))
print("Isolated source rows:", spark.table(test_source_table).count())

In [0]:
from pyspark.sql import functions as F

# Safety checks: all isolated test tables must still be empty.
for table_name in [
    test_source_table,
    test_valid_table,
    test_quarantine_table
]:
    if spark.table(table_name).count() != 0:
        raise RuntimeError(
            f"Safety stop: {table_name} is not empty."
        )

# Select one valid JSON event from the existing raw table.
template = (
    spark.table(streaming_raw_table)
    .withColumn("parsed_taxi", F.from_json("raw_value", taxi_schema))
    .filter(F.col("parsed_taxi").isNotNull())
    .filter(F.col("parsed_taxi.VendorID").isNotNull())
    .filter(F.col("parsed_taxi.total_amount").isNotNull())
    .select("raw_value", "kafka_timestamp")
    .limit(1)
)

if template.count() != 1:
    raise RuntimeError("Could not find exactly one valid template event.")

# Assign a synthetic identity distinct from real Kafka identities.
synthetic_event = template.select(
    F.lit("taxi-trips-incremental-test").alias("topic"),
    F.lit(999).cast("int").alias("kafka_partition"),
    F.lit(0).cast("long").alias("kafka_offset"),
    F.col("kafka_timestamp"),
    F.lit("synthetic-incremental-test").alias("message_key"),
    F.col("raw_value"),
    F.current_timestamp().alias("captured_at")
)

# Write exclusively to the isolated test source.
synthetic_event.write.format("delta").mode("append").saveAsTable(
    test_source_table
)

print("Test source rows:", spark.table(test_source_table).count())
print("Production raw rows:", spark.table(streaming_raw_table).count())

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Safety checks: output tables must be empty.
if spark.table(test_valid_table).count() != 0:
    raise RuntimeError("Test valid table is not empty.")

if spark.table(test_quarantine_table).count() != 0:
    raise RuntimeError("Test quarantine table is not empty.")

# Ensure the isolated checkpoints are not already present.
for path in test_checkpoints.values():
    try:
        dbutils.fs.ls(path)
        raise RuntimeError(
            f"Safety stop: checkpoint already exists: {path}"
        )
    except RuntimeError:
        raise
    except Exception as e:
        message = str(e).lower()
        if not any(term in message for term in [
            "filenotfound", "does not exist", "path_not_found"
        ]):
            raise RuntimeError(
                f"Could not verify checkpoint path {path}: {e}"
            )

# Read the isolated Delta source as a streaming DataFrame.
df_incremental_test_stream = (
    spark.readStream
    .format("delta")
    .table(test_source_table)
)

print("Isolated stream schema:")
df_incremental_test_stream.printSchema()
print("Source rows available:", spark.table(test_source_table).count())
print("Safety checks passed.")

In [0]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F

required_fields = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "total_amount"
]

def process_test_valid_batch(batch_df, batch_id):
    parsed = batch_df.withColumn(
        "taxi", F.from_json(F.col("raw_value"), taxi_schema)
    )

    is_valid = F.lit(True)
    for field_name in required_fields:
        is_valid = is_valid & F.col(f"taxi.{field_name}").isNotNull()

    valid_batch = (
        parsed.filter(is_valid)
        .select(
            "topic", "kafka_partition", "kafka_offset",
            "kafka_timestamp", "message_key", "raw_value",
            "captured_at", "taxi.*"
        )
    )

    target = DeltaTable.forName(spark, test_valid_table)
    (
        target.alias("target")
        .merge(
            valid_batch.alias("source"),
            """
            target.topic = source.topic
            AND target.kafka_partition = source.kafka_partition
            AND target.kafka_offset = source.kafka_offset
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Test valid batch {batch_id} processed.")

def process_test_quarantine_batch(batch_df, batch_id):
    parsed = batch_df.withColumn(
        "taxi", F.from_json(F.col("raw_value"), taxi_schema)
    )

    missing_required = F.lit(False)
    for field_name in required_fields:
        missing_required = missing_required | F.col(
            f"taxi.{field_name}"
        ).isNull()

    quarantine_batch = (
        parsed.withColumn(
            "validation_status",
            F.when(
                F.col("taxi").isNull(),
                F.lit("QUARANTINE_INVALID_JSON")
            ).when(
                missing_required,
                F.lit("QUARANTINE_MISSING_REQUIRED_FIELDS")
            )
        )
        .filter(F.col("validation_status").isNotNull())
        .select(
            "topic", "kafka_partition", "kafka_offset",
            "kafka_timestamp", "message_key", "raw_value",
            "captured_at", "validation_status"
        )
    )

    target = DeltaTable.forName(spark, test_quarantine_table)
    (
        target.alias("target")
        .merge(
            quarantine_batch.alias("source"),
            """
            target.topic = source.topic
            AND target.kafka_partition = source.kafka_partition
            AND target.kafka_offset = source.kafka_offset
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Test quarantine batch {batch_id} processed.")

print("Isolated valid and quarantine processors defined.")


In [0]:
# Recheck output tables before starting.
if spark.table(test_valid_table).count() != 0:
    raise RuntimeError("Safety stop: test valid table is not empty.")

if spark.table(test_quarantine_table).count() != 0:
    raise RuntimeError("Safety stop: test quarantine table is not empty.")

# Start valid-event processing.
test_valid_query = (
    df_incremental_test_stream.writeStream
    .foreachBatch(process_test_valid_batch)
    .option("checkpointLocation", test_checkpoints["valid"])
    .trigger(availableNow=True)
    .start()
)

test_valid_query.awaitTermination()

print("Valid test stream completed.")
print("Status:", test_valid_query.status)
print("Progress:", test_valid_query.lastProgress)

In [0]:
test_quarantine_query = (
    df_incremental_test_stream.writeStream
    .foreachBatch(process_test_quarantine_batch)
    .option("checkpointLocation", test_checkpoints["quarantine"])
    .trigger(availableNow=True)
    .start()
)

test_quarantine_query.awaitTermination()

print("Quarantine test stream completed.")
print("Status:", test_quarantine_query.status)
print("Progress:", test_quarantine_query.lastProgress)

In [0]:
from pyspark.sql import functions as F

identity = (
    (F.col("topic") == "taxi-trips-incremental-test")
    & (F.col("kafka_partition") == 999)
    & (F.col("kafka_offset") == 0)
)

test_source_count = spark.table(test_source_table).count()
test_valid_count = spark.table(test_valid_table).count()
test_quarantine_count = spark.table(test_quarantine_table).count()

test_valid_unique = (
    spark.table(test_valid_table)
    .select("topic", "kafka_partition", "kafka_offset")
    .distinct()
    .count()
)

synthetic_valid_count = spark.table(test_valid_table).filter(identity).count()
synthetic_quarantine_count = spark.table(test_quarantine_table).filter(identity).count()

print("ISOLATED TEST")
print("Source rows:", test_source_count)
print("Valid output rows:", test_valid_count)
print("Unique valid identities:", test_valid_unique)
print("Quarantine output rows:", test_quarantine_count)
print("Synthetic identity in valid:", synthetic_valid_count)
print("Synthetic identity in quarantine:", synthetic_quarantine_count)

print("\nORIGINAL PHASE 4 TABLES")
print("Raw:", spark.table(streaming_raw_table).count())
print("Valid:", spark.table(streaming_valid_table).count())
print("Quarantine:", spark.table(streaming_quarantine_table).count())

assert test_source_count == 1
assert test_valid_count == 1
assert test_valid_unique == 1
assert test_quarantine_count == 0
assert synthetic_valid_count == 1
assert synthetic_quarantine_count == 0

assert spark.table(streaming_raw_table).count() == 302
assert spark.table(streaming_valid_table).count() == 300
assert spark.table(streaming_quarantine_table).count() == 2

print("\nPASS: Incremental test succeeded; original Phase 4 counts are unchanged.")